# 1 Module, Globals und Daten

## 1.1 Module

In [ ]:
##### IMPORTS #####

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, learning_curve
import sklearn.metrics as skm
import xgboost as xgb
from xgboost import XGBClassifier

import matplotlib.pyplot as pyplot

## 1.2 Globals

In [ ]:
##### Globals #####
dataPath = 'datasets/gyro/'                         # Set location of dataset


## 1.3 Daten

In [ ]:
##### Data Import and Splitting #####
data = pd.read_csv(dataPath + 'gyro_mobile.csv')    # Dataset is imbalanced with only ~1.7% of all labels being 0's
data = data.drop(columns='timestamp')
xdata = data.iloc[:,:6]
ydata = data.iloc[:,6:]

xtrain, xtest, ytrain, ytest = train_test_split( 
    xdata,
    ydata,
    random_state=0,
    train_size=0.33,
    stratify=ydata                                  # Preserve label imbalance across train- and test datasets
)

ev_test = [(xtest,ytest)]
ev_train = [(xtrain,ytrain)]
ev_both = [(xtest,ytest),(xtrain,ytrain)]

## 1.4 Funktionen

In [ ]:
def plotReport(model):
    global xtest, ytest
    yhat = model.predict(xtest)
    print(f'Accuracy Score: \t\t{skm.accuracy_score(ytest, yhat)}')          # 0.9828310161425772 stratified | 0.981664644956611  naive  | 0.9534384622562284 scale_pos_weight = len(0s)/len(1s)
    print(f'Balanced Accuracy-Score: \t{skm.balanced_accuracy_score(ytest, yhat)}') # 0.5990495790838934 stratified | 0.5770402535045962 naive  | 0.8314600386751905 scale_pos_weight = len(0s)/len(1s)
    print()
    print(skm.classification_report(ytest,yhat, labels=[0,1], target_names=['0: standing','1: walking']))

    skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap='Blues', normalize='true')
    pyplot.title('')
    pyplot.show()

    skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap="Blues" ,values_format='d')
    pyplot.show()

def plotLearningCurves(model):
    
    # https://xgboosting.com/xgboost-plot-learning-curve/

    global xtest, ytest
    
    # Calculate learning curves
    train_sizes, train_scores, test_scores = learning_curve(
    estimator=model, X=xdata, y=ydata, cv=5, scoring='accuracy',
    train_sizes=np.linspace(0.75, 1.0, 10))

    # Calculate mean and standard deviation of scores
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)

    # Plot learning curves
    pyplot.figure(figsize=(8, 6))
    pyplot.plot(train_sizes, train_mean, color='blue', marker='o', label='Training accuracy')
    pyplot.fill_between(train_sizes, train_mean + train_std, train_mean - train_std, alpha=0.15, color='blue')
    pyplot.plot(train_sizes, test_mean, color='green', marker='+', label='Validation accuracy')
    pyplot.fill_between(train_sizes, test_mean + test_std, test_mean - test_std, alpha=0.15, color='green')
    pyplot.title('Learning Curves')
    pyplot.xlabel('Training set size')
    pyplot.ylabel('Accuracy')
    pyplot.grid()
    pyplot.legend(loc='lower right')
    pyplot.show()

def plotFeatureImportances(model):    
    iScores = model.feature_importances_
    fNames = model.feature_names_in_

    fig,ax = pyplot.subplots()
    ax.bar(fNames, iScores)
    ax.set_ylabel("Importance Scores")
    ax.set_xlabel("Features")

    pyplot.show()

def plotGeneralizationCurves(model, custom_names = None):
    results = model.evals_result()
    names = list(model.evals_result())
    lossValue = list(results[names[0]])[0]
    
    if custom_names == None:
        for i in names:
            pyplot.plot(results[i][lossValue], label=i)
    else:
        name_index = 0
        for i in names:
            pyplot.plot(results[i][lossValue], label=custom_names[name_index])
            name_index += 1

    pyplot.ylabel(lossValue)
    pyplot.xlabel('Iterations')
    pyplot.legend()
    pyplot.show()

def checkOverfitting(model):
    global xtrain, ytrain, xtest, ytest
    train_acc = skm.accuracy_score(ytrain, model.predict(xtrain))
    test_acc = skm.accuracy_score(ytest, model.predict(xtest))
    print(f'Train-Test-Difference: {train_acc - test_acc}')

# 2 Modell

## 2.1 Training

### 2.2.1 Naive Evaluation

In [ ]:
clf_naive = XGBClassifier(
    objective = "binary:logistic",
    tree_method = 'exact',
    n_estimators = 100000,
    early_stopping_rounds = 100,
    )


clf_naive.fit(
    xtrain, ytrain,
    eval_set = ev_test,
    verbose = 0
)

print(f'Best Iteration based on Test-Data: {clf_naive.best_iteration}')
plotReport(clf_naive)
plotGeneralizationCurves(clf_naive)

new_n_estimators = clf_naive.early_stopping_rounds + clf_naive.best_iteration

clf_naive.set_params(
    n_estimators = new_n_estimators,
    early_stopping_rounds = None
)
clf_naive.fit(
    xtrain, ytrain,
    eval_set = ev_both,
    verbose = 0
)
plotGeneralizationCurves(clf_naive)

### 2.2.2 Adjusted for Imbalance: Scaled Positives

[XGBoost for Imbalanced Classification](https://xgboosting.com/xgboost-scale_pos_weight-vs-sample_weight-for-imbalanced-classification/)

In [ ]:
clf_scaled = XGBClassifier(
    objective = "binary:logistic",
    tree_method = 'exact',
    scale_pos_weight = len(data[data['Activity']==0]) / len(data[data['Activity']==1]), # Increase weight of minority class
    n_estimators = 100000,
    early_stopping_rounds = 100,
)

clf_scaled.fit(xtrain,ytrain,
               eval_set=ev_test,
               verbose = 0)

print(f'Best Iteration based on Test-Data: {clf_scaled.best_iteration}')
plotReport(clf_scaled)
plotGeneralizationCurves(clf_scaled)

new_n_estimators = clf_scaled.early_stopping_rounds + clf_scaled.best_iteration

bestIteration = clf_scaled.best_iteration

clf_scaled.set_params(
    n_estimators = new_n_estimators,
    early_stopping_rounds = None
)
clf_scaled.fit(
    xtrain, ytrain,
    eval_set = ev_both,
    verbose = 0
)
plotGeneralizationCurves(clf_scaled)

### 2.2.3 Scaled Best Iteration

In [ ]:
clf_scaled_bi = XGBClassifier(
    objective = "binary:logistic",
    tree_method = 'exact',
    scale_pos_weight = len(data[data['Activity']==0]) / len(data[data['Activity']==1]), # Increase weight of minority class
    n_estimators = bestIteration,
    )

clf_scaled_bi.fit(
    xtrain, ytrain,
    eval_set = ev_both,
    verbose = 0
)
print(f'Amount of Estimators: \t\t{bestIteration}')
plotReport(clf_scaled_bi)
plotGeneralizationCurves(clf_scaled_bi)

# Cross-Validated Learning Curves

Besser geeignet, wenn die Klassen nicht so pervers unausgeglichen sind

In [ ]:
es_clf = XGBClassifier()

es_clf.set_params(    
    tree_method = 'exact',
    scale_pos_weight = len(data[data['Activity']==0]) / len(data[data['Activity']==1]), # Increase weight of minority class
    n_estimators = 10000,
    early_stopping_rounds = 50,
    max_depth = 2,
    random_state = 0
)

es_clf.fit(
    xtrain,ytrain,
    eval_set = evalset,
    verbose = 0
)

printLossCurves(es_clf,["Training","Test"])

# Sonstiges/Eingestellt

## Skalierung der Daten

Es wurde neben dem eigentlichem Datensatz auch Modell mit einem Datensatz trainiert, bei dem die Daten nicht skaliert worden sind.
Die daraus resultierende Accuracy war identisch mit der unskalierten Version.


In [ ]:
# ##### Scaled comparison #####

# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# xdata_scaled = scaler.fit_transform(xdata)

# xtrain_scaled, xtest_scaled, ytrain_scaled, ytest_scaled = train_test_split( 
#     xdata_scaled,
#     ydata,
#     random_state=0,
#     stratify=ydata                                  # Preserve label imbalance across train- and test datasets
# )

# model_scaled = XGBClassifier(
#     objective='binary:logistic'
# )
# model_scaled.fit(xtrain_scaled,ytrain)

# yhat_scaled = model_scaled.predict(xtest_scaled)
# print(accuracy_score(ytest_scaled, yhat_scaled))    # 0.9834958739684921

## C-Reports und C-Matrizen für unterschiedliche Tree_Methods

### Raw

In [ ]:
# model.set_params(tree_method = 'approx')
# model.fit(xtrain, ytrain,
#           eval_set=evalset,
#           verbose = 0)

# ##### Predictions and Accuracy #####
# yhat = model.predict(xtest)

# print(f'Accuracy Score: \t\t{skm.accuracy_score(ytest, yhat)}')          # 0.9828310161425772 stratified | 0.981664644956611  naive  | 0.9534384622562284 scale_pos_weight = len(0s)/len(1s)
# print(f'Balanced Accuracy-Score: \t{skm.balanced_accuracy_score(ytest, yhat)}') # 0.5990495790838934 stratified | 0.5770402535045962 naive  | 0.8314600386751905 scale_pos_weight = len(0s)/len(1s)
# print()
# print(skm.classification_report(ytest,yhat, labels=[0,1], target_names=['0: standing','1: walking']))

# skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap='Blues', values_format='d')
# pyplot.show()


In [ ]:
# model.set_params(tree_method = 'hist')
# model.fit(xtrain, ytrain,
#           eval_set=evalset,
#           verbose = 0)

# ##### Predictions and Accuracy #####
# yhat = model.predict(xtest)

# print(f'Accuracy Score: \t\t{skm.accuracy_score(ytest, yhat)}')          # 0.9828310161425772 stratified | 0.981664644956611  naive  | 0.9534384622562284 scale_pos_weight = len(0s)/len(1s)
# print(f'Balanced Accuracy-Score: \t{skm.balanced_accuracy_score(ytest, yhat)}') # 0.5990495790838934 stratified | 0.5770402535045962 naive  | 0.8314600386751905 scale_pos_weight = len(0s)/len(1s)
# print()
# print(skm.classification_report(ytest,yhat, labels=[0,1], target_names=['0: standing','1: walking']))

# skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap='Blues', values_format='d')
# pyplot.show()

### Angepasst

In [ ]:
# model.set_params(    
#     tree_method = 'approx',
#     scale_pos_weight = len(data[data['Activity']==0]) / len(data[data['Activity']==1]), # Increase weight of minority class
#     max_delta_step = 1                                                                  # Limit the maximum change in the predictions)
# )

# model.fit(xtrain,ytrain,
#           eval_set=evalset,
#           verbose = 0)

# yhat = model.predict(xtest)

# print(f'Accuracy Score: \t\t{skm.accuracy_score(ytest, yhat)}')          # 0.9828310161425772 stratified | 0.981664644956611  naive  | 0.9534384622562284 scale_pos_weight = len(0s)/len(1s)
# print(f'Balanced Accuracy-Score: \t{skm.balanced_accuracy_score(ytest, yhat)}') # 0.5990495790838934 stratified | 0.5770402535045962 naive  | 0.8314600386751905 scale_pos_weight = len(0s)/len(1s)
# print()
# print(skm.classification_report(ytest,yhat, labels=[0,1], target_names=['0: standing','1: walking']))

# skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap='Blues', values_format='d')
# pyplot.show()


In [ ]:
# model.set_params(    
#     tree_method = 'hist',
#     scale_pos_weight = len(data[data['Activity']==0]) / len(data[data['Activity']==1]), # Increase weight of minority class
#     max_delta_step = 0
# )

# model.fit(xtrain,ytrain,
#           eval_set=evalset,
#           verbose = 0)

# yhat = model.predict(xtest)

# print(f'Accuracy Score: \t\t{skm.accuracy_score(ytest, yhat)}')          # 0.9828310161425772 stratified | 0.981664644956611  naive  | 0.9534384622562284 scale_pos_weight = len(0s)/len(1s)
# print(f'Balanced Accuracy-Score: \t{skm.balanced_accuracy_score(ytest, yhat)}') # 0.5990495790838934 stratified | 0.5770402535045962 naive  | 0.8314600386751905 scale_pos_weight = len(0s)/len(1s)
# print()
# print(skm.classification_report(ytest,yhat, labels=[0,1], target_names=['0: standing','1: walking']))

# skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap='Blues', values_format='d')
# pyplot.show()


### Validieren, dass tree_method "auto" == "hist"

In [ ]:
# model = XGBClassifier()
# model.set_params(tree_method = 'auto',
#                  objective = "binary:logistic",
#                  scale_pos_weight = len(data[data['Activity']==0]) / len(data[data['Activity']==1]))
# model.fit(xtrain, ytrain)

# yhat = model.predict(xtest)

# print(f'Accuracy Score: \t\t{skm.accuracy_score(ytest, yhat)}')          
# print(f'Balanced Accuracy-Score: \t{skm.balanced_accuracy_score(ytest, yhat)}')

# # Accuracy Score: 		    0.9534384622562284
# # Balanced Accuracy-Score: 	0.8314600386751905

# C-Matrix Optionen

In [ ]:
# Gibt das Verhältnis von richtig und falsch bzgl. einer Vorhersage an
# skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap='Blues', normalize='pred')
# pyplot.title('Normalize = Pred')
# pyplot.show()

# Gibt das Verhältnis der jeweiligen TP/FP/TN/FN aus allen Vorhersagen an
# skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap='Blues', normalize='all')
# pyplot.title('Normalize = All')
# pyplot.show()